# Validação da silver contra a landing e o gabarito

Confere se a silver e a quarentena reproduzem o que o simulador **sabe** que fez (`ops.simulator_chaos_log`) e o que existe na landing. Cada verificação grava uma linha em `olist.quality.validacao_resultados` (esperado × obtido × ok), e o notebook **falha no final** se alguma estiver reprovada.

**Verificações (desenho S9)**

| # | Verificação | Entidades | Esperado |
|---|---|---|---|
| 1 | `bronze_igual_landing` | todas | linhas da bronze = linhas dos arquivos da landing |
| 2 | `balanco_de_linhas` | com chave | linhas da bronze = silver + quarentena + duplicatas calculadas na landing |
| 3 | `silver_igual_chaves_da_landing` | com chave | linhas da silver = chaves distintas da landing, menos as quarentenadas |
| 4 | `silver_sem_chave_duplicada` | com chave | 0 chaves repetidas |
| 5 | `gabarito_corrupt_file` | todas | linhas corrompidas do gabarito = linhas na quarentena por `sem_linha_corrompida` |
| 6 | `gabarito_bad_value_corrigido` | `order_items` | todo `bad_value` do gabarito está na silver, marcado em `_dq_correcao` |
| 7 | `gabarito_new_column` | `order_payments` | coluna `payment_gateway` existe e não é nula depois da data em que surgiu |
| 8 | `gabarito_late_arrival_presente` | `order_items`, `order_status_events` | todo registro atrasado está na silver |
| 9 | `gabarito_out_of_order_correto` | `order_status_events` | eventos de arquivos fora de ordem estão na silver com o `event_ts` da bronze |
| 10 | `idempotencia_silver_estavel` | todas | com a bronze igual, a silver não muda entre duas validações |
| 11 | `cdc_ultima_versao` | `customers`, `sellers` | a silver guarda, por chave, a versão de maior `change_sequence` da landing |
| 12 | `cdc_exclusoes_marcadas` | `customers`, `sellers` | chaves cuja última versão é `DELETE` = linhas da silver com `operation = 'DELETE'` |

`duplicate_record` e `file_resend` são cobertos pelas verificações 2 a 4: o gabarito conta arquivos (`file_resend`), não registros, então a comparação usa as linhas efetivamente repetidas, calculadas na landing de forma independente.

**Como provar a idempotência (item 10):** rode a validação, rode o pipeline de novo sem dados novos e rode a validação outra vez. A segunda execução compara com a primeira.

In [0]:
# O QUE FAZ: define as constantes (nomes de tabelas, chaves de cada entidade com chave natural,
#   entidades de referência) e cria a lista onde as verificações vão se acumulando.
#
# RECURSOS:
# - uuid.uuid4(): identificador aleatório único desta validação (agrupa as linhas dela).
# - dict {entidade: (colunas da chave)}: mesma chave natural usada no pipeline (desenho S3).
# - tuple: lista que não muda.
# - Nota: a chave está repetida do notebook do pipeline porque os notebooks não compartilham
#   código; a Fase 7 move isso para um módulo único.
import uuid
from functools import reduce

from pyspark.sql import functions as F

CATALOG = "olist"
LANDING = f"/Volumes/{CATALOG}/landing/raw_data"
GABARITO = f"{CATALOG}.ops.simulator_chaos_log"
TABELA_RESULTADOS = f"{CATALOG}.quality.validacao_resultados"

VALIDACAO_ID = str(uuid.uuid4())

CHAVES = {
    "orders": ("order_id",),
    "order_items": ("order_id", "order_item_id"),
    "order_payments": ("order_id", "payment_sequential"),
    "order_reviews": ("review_id", "order_id"),
    "order_status_events": ("order_id", "event_sequence"),
    "customers": ("customer_id",),
    "sellers": ("seller_id",),
}
REFERENCIA = ("products", "geolocation", "category_translation")
TODAS = (*CHAVES, *REFERENCIA)

resultados = []  # (entidade, verificacao, esperado, obtido, ok, detalhe)
print(f"validacao_id: {VALIDACAO_ID}")

## Tabela de resultados e funções auxiliares

`registrar` guarda uma verificação (ok = esperado igual ao obtido; `esperado` vazio significa "só registro", como as fotos de contagem usadas na idempotência). As demais funções montam textos e colunas e **não** gravam nada.

In [0]:
# O QUE FAZ: cria a tabela de resultados (se não existir) e declara as funções auxiliares.
#
# FUNÇÕES DECLARADAS:
# - registrar(entidade, verificacao, esperado, obtido, detalhe): acrescenta uma linha em
#   "resultados". ok = (esperado == obtido); sem esperado (None), ok fica nulo = "não se aplica".
# - chave_texto(colunas): coluna de texto "valor1|valor2", o mesmo formato do record_key do gabarito.
# - com_chave(colunas): condição "todas as colunas da chave não são nulas".
# - tabela(camada, entidade): nome completo da tabela (bronze, silver ou quarentena).
#
# FUNÇÕES NATIVAS:
# - F.concat_ws("|", ...): junta colunas com separador. F.col(c).cast("string"): converte para texto.
# - functools.reduce(função, lista): aplica a função acumulando; aqui combina condições com AND.
# - F.col(c).isNotNull(): teste de não nulo.
def registrar(entidade, verificacao, esperado, obtido, detalhe=""):
    ok = None if esperado is None else (esperado == obtido)
    resultados.append((entidade, verificacao, esperado, obtido, ok, detalhe))
    marca = "  " if ok is None else ("OK" if ok else "XX")
    print(f"[{marca}] {entidade:22} {verificacao:34} esperado={esperado} obtido={obtido} {detalhe}")


def chave_texto(colunas):
    return F.concat_ws("|", *[F.col(c).cast("string") for c in colunas])


def com_chave(colunas):
    return reduce(lambda a, b: a & b, [F.col(c).isNotNull() for c in colunas])


def tabela(camada, entidade):
    if camada == "quarentena":
        return f"{CATALOG}.quality.{entidade}_quarentena"
    return f"{CATALOG}.{camada}.{entidade}"


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_RESULTADOS} (
  validacao_id  STRING    COMMENT 'Identificador da execução da validação',
  verificado_em TIMESTAMP COMMENT 'Momento da verificação',
  entidade      STRING    COMMENT 'Entidade verificada',
  verificacao   STRING    COMMENT 'Nome da verificação (desenho S9)',
  esperado      BIGINT    COMMENT 'Valor esperado; nulo quando a linha é só um registro de contagem',
  obtido        BIGINT    COMMENT 'Valor obtido no lakehouse',
  ok            BOOLEAN   COMMENT 'Verdadeiro se esperado = obtido; nulo se não se aplica',
  detalhe       STRING    COMMENT 'Texto explicativo'
)
COMMENT 'Resultado da validação da silver contra a landing e o gabarito do simulador (desenho S9).'
""")

## Verificações 1 a 4 — contagens e chaves

Para cada entidade, a landing é lida **de novo, direto dos arquivos JSON**, sem passar pela bronze. Isso torna a conferência independente: se a bronze perdesse ou duplicasse linhas, a verificação 1 acusaria.

- *Duplicatas calculadas na landing* = linhas com chave − chaves distintas.
- *Chaves quarentenadas* saem do esperado da silver, porque um registro reprovado não deve estar nela.
- Entidades de referência (sem chave) só entram nas verificações 1 e no balanço bronze = silver + quarentena.

In [0]:
# O QUE FAZ: para cada entidade, lê a landing, calcula as contagens de forma independente e registra as
#   verificações 1 a 4 (e o balanço das entidades de referência). Também guarda as "fotos" de
#   contagem da bronze e da silver usadas na verificação 10 (idempotência).
#
# FUNÇÕES NATIVAS:
# - spark.read.option(chave, valor).json(pasta): lê os arquivos JSON de uma pasta.
#     * primitivesAsString=true: lê números como texto, igual à bronze.
#     * columnNameOfCorruptRecord: coluna que recebe o texto da linha que não é JSON válido.
#     * A pasta tem subpastas sim_date=AAAA-MM-DD; o Spark lê todas e cria a coluna sim_date.
# - spark.read.text(pasta): lê cada linha como texto (para só contar linhas de referência).
# - DataFrame.where(cond) / select / distinct / count: filtrar, escolher colunas, valores únicos, contar.
# - DataFrame.join(outro, "k", "left_anti"): linhas de um lado que NÃO aparecem no outro.
# - DataFrame.groupBy(cols).count(): conta linhas por chave; usado para achar chaves repetidas.
for entidade in TODAS:
    bronze = spark.table(tabela("bronze", entidade)).count()
    silver_df = spark.table(tabela("silver", entidade))
    silver = silver_df.count()
    quarentena_df = spark.table(tabela("quarentena", entidade))
    quarentena = quarentena_df.count()

    # fotos para a verificação 10
    registrar(entidade, "foto_bronze", None, bronze)
    registrar(entidade, "foto_silver", None, silver)

    if entidade in CHAVES:
        chaves = CHAVES[entidade]
        landing = (
            spark.read.option("primitivesAsString", "true")
            .option("columnNameOfCorruptRecord", "_corrupt_record")
            .json(f"{LANDING}/{entidade}")
        )
        linhas_landing = landing.count()
        landing_com_chave = landing.where(com_chave(chaves))
        linhas_com_chave = landing_com_chave.count()
        chaves_landing = landing_com_chave.select(chave_texto(chaves).alias("k")).distinct()
        n_chaves = chaves_landing.count()
        duplicatas = linhas_com_chave - n_chaves

        chaves_quarentena = (
            quarentena_df.where(com_chave(chaves)).select(chave_texto(chaves).alias("k")).distinct()
        )
        esperado_silver = chaves_landing.join(chaves_quarentena, "k", "left_anti").count()
        repetidas = silver_df.groupBy(*chaves).count().where("count > 1").count()

        registrar(entidade, "bronze_igual_landing", linhas_landing, bronze)
        registrar(
            entidade, "balanco_de_linhas", bronze, silver + quarentena + duplicatas,
            f"silver {silver} + quarentena {quarentena} + duplicatas {duplicatas}",
        )
        registrar(entidade, "silver_igual_chaves_da_landing", esperado_silver, silver)
        registrar(entidade, "silver_sem_chave_duplicada", 0, repetidas)
    else:
        linhas_landing = spark.read.text(f"{LANDING}/{entidade}").count()
        registrar(entidade, "bronze_igual_landing", linhas_landing, bronze)
        registrar(
            entidade, "balanco_de_linhas", bronze, silver + quarentena,
            f"silver {silver} + quarentena {quarentena}",
        )

## Verificações 5 a 9 — cada falha do gabarito tem o destino esperado

| Falha no gabarito | Destino esperado |
|---|---|
| `corrupt_file` | quarentena, motivo `sem_linha_corrompida` |
| `bad_value` | silver, **corrigido** e marcado em `_dq_correcao` (decisão D1) |
| `new_column` | `payment_gateway` existe e é preenchida a partir da data em que surgiu |
| `late_arrival` | o registro atrasado está na silver |
| `out_of_order` | os eventos do arquivo fora de ordem estão na silver com o `event_ts` correto |

A chave do gabarito (`record_key`) usa o mesmo formato `valor1|valor2` de `chave_texto`.

In [0]:
# O QUE FAZ: confere, uma falha do gabarito por vez, se o destino é o esperado (verificações 5 a 9).
#
# FUNÇÕES NATIVAS:
# - spark.table(nome): abre uma tabela como DataFrame.
# - F.col("motivo").contains("texto"): verdadeiro se o texto aparece dentro da coluna.
# - DataFrame.agg(F.min(coluna)).first()[0]: menor valor da coluna, como valor Python.
# - "payment_gateway" in df.columns: testa se a coluna existe.
# - F.expr("try_cast(x AS TIPO)"): conversão que devolve nulo se falhar.
# - DataFrame.join(outro, cols, "left_semi"): mantém só as linhas que têm par no outro lado.
# - DataFrame.join(outro, cols, "left"): mantém todas as linhas da esquerda.
gab = spark.table(GABARITO)


def chaves_do_gabarito(entidade, tipo):
    """Chaves (record_key) distintas que o simulador marcou com essa falha."""
    return (
        gab.where((F.col("entity") == entidade) & (F.col("chaos_type") == tipo) & F.col("record_key").isNotNull())
        .select(F.col("record_key").alias("k"))
        .distinct()
    )


# 5. arquivo corrompido -> quarentena (soma das entidades)
esperado = gab.where(F.col("chaos_type") == "corrupt_file").count()
obtido = sum(
    spark.table(tabela("quarentena", e)).where(F.col("motivo").contains("sem_linha_corrompida")).count()
    for e in TODAS
)
registrar("todas", "gabarito_corrupt_file", esperado, obtido)

# 6. valor fora do padrão -> corrigido e marcado
silver_itens = spark.table(tabela("silver", "order_items"))
corrigidos = silver_itens.where(F.col("_dq_correcao") == "price_virgula_decimal").select(
    chave_texto(CHAVES["order_items"]).alias("k")
)
bad = chaves_do_gabarito("order_items", "bad_value")
registrar("order_items", "gabarito_bad_value_corrigido", bad.count(), bad.join(corrigidos, "k", "left_semi").count())
registrar("order_items", "correcao_sem_excesso", bad.count(), corrigidos.distinct().count(), "correções que não estão no gabarito")

# 7. coluna nova -> payment_gateway existe e é preenchida depois da data em que surgiu
silver_pag = spark.table(tabela("silver", "order_payments"))
data_surgiu = gab.where((F.col("entity") == "order_payments") & (F.col("chaos_type") == "new_column")).agg(F.min("sim_date")).first()[0]
nulos_depois = (
    silver_pag.where((F.col("sim_date") >= F.lit(data_surgiu)) & F.col("payment_gateway").isNull()).count()
    if "payment_gateway" in silver_pag.columns else -1
)
registrar("order_payments", "gabarito_new_column", 0, nulos_depois, f"coluna surgiu em {data_surgiu}; -1 = coluna ausente")

# 8. chegada atrasada -> registro presente na silver
for entidade in ("order_items", "order_status_events"):
    atrasados = chaves_do_gabarito(entidade, "late_arrival")
    na_silver = spark.table(tabela("silver", entidade)).select(chave_texto(CHAVES[entidade]).alias("k"))
    registrar(entidade, "gabarito_late_arrival_presente", 0, atrasados.join(na_silver, "k", "left_anti").count(), f"{atrasados.count()} registros atrasados")

# 9. arquivo fora de ordem -> event_ts da silver igual ao da bronze, para todos os eventos desses arquivos
dias_fora_de_ordem = (
    gab.where((F.col("entity") == "order_status_events") & (F.col("chaos_type") == "out_of_order"))
    .select(F.col("sim_date").cast("string").alias("sim_date"))
    .distinct()
)
bronze_ev = (
    spark.table(tabela("bronze", "order_status_events"))
    .join(dias_fora_de_ordem, "sim_date", "left_semi")
    .select(
        "order_id",
        F.col("event_sequence").cast("int").alias("event_sequence"),
        F.expr("try_cast(event_ts AS TIMESTAMP)").alias("ts_bronze"),
    )
    .distinct()
)
silver_ev = spark.table(tabela("silver", "order_status_events")).select(
    "order_id", "event_sequence", F.col("event_ts").alias("ts_silver")
)
divergentes = (
    bronze_ev.join(silver_ev, ["order_id", "event_sequence"], "left")
    .where(F.col("ts_silver").isNull() | (F.col("ts_silver") != F.col("ts_bronze")))
    .count()
)
registrar("order_status_events", "gabarito_out_of_order_correto", 0, divergentes, f"{bronze_ev.count()} eventos conferidos")

## Verificações 11 e 12 — feeds de mudança (`customers` e `sellers`)

Esses dois feeds trazem `operation` (INSERT, UPDATE, DELETE) e `change_sequence` (1, 2, 3: a versão da linha na chave). A silver da etapa 2.5 guarda **uma linha por chave, a de maior `change_sequence`**, e mantém `operation`; o histórico completo fica para a Fase 3.

- **11:** para cada chave, o `change_sequence` e o `operation` da silver têm que ser os da última versão da landing. Conta as chaves em que divergem (esperado 0).
- **12:** o número de chaves cuja última versão é `DELETE` na landing tem que ser igual ao número de linhas da silver com `operation = 'DELETE'`. Como a exclusão não apaga a linha da silver (isso é tema da Fase 3 e da Fase 6), a contagem tem que bater.

In [0]:
# O QUE FAZ: relê a landing de customers e sellers, descobre a última versão de cada chave e compara
#   com a silver (verificações 11 e 12).
#
# FUNÇÕES NATIVAS:
# - Window.partitionBy(chave).orderBy(coluna.desc()): janela por chave, da versão mais nova para
#   a mais antiga.
# - F.row_number().over(janela): numera as linhas dentro da janela; a 1 é a última versão.
# - F.col("change_sequence").cast("int"): a landing é lida como texto; converte para comparar.
# - DataFrame.join(outro, cols, "left"): mantém todas as linhas da esquerda.
# - DataFrame.where(cond).count(): conta as linhas que cumprem a condição.
from pyspark.sql import Window

for entidade in ("customers", "sellers"):
    chave = CHAVES[entidade][0]
    landing = (
        spark.read.option("primitivesAsString", "true")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .json(f"{LANDING}/{entidade}")
        .where(F.col(chave).isNotNull())
    )
    janela = Window.partitionBy(chave).orderBy(F.col("change_sequence").cast("int").desc())
    ultima = (
        landing.withColumn("n", F.row_number().over(janela))
        .where("n = 1")
        .select(
            chave,
            F.col("change_sequence").cast("int").alias("seq_landing"),
            F.col("operation").alias("op_landing"),
        )
    )
    silver_cdc = spark.table(tabela("silver", entidade)).select(
        chave, F.col("change_sequence").alias("seq_silver"), F.col("operation").alias("op_silver")
    )
    divergentes = (
        ultima.join(silver_cdc, chave, "left")
        .where(
            F.col("seq_silver").isNull()
            | (F.col("seq_silver") != F.col("seq_landing"))
            | (F.col("op_silver") != F.col("op_landing"))
        )
        .count()
    )
    registrar(entidade, "cdc_ultima_versao", 0, divergentes, f"{ultima.count()} chaves conferidas")

    esperado = ultima.where("op_landing = 'DELETE'").count()
    obtido = silver_cdc.where("op_silver = 'DELETE'").count()
    registrar(entidade, "cdc_exclusoes_marcadas", esperado, obtido, "chaves com última versão DELETE")

## Verificação 10 — idempotência

Compara as contagens desta validação com as da **validação anterior**. Se a bronze não mudou entre as duas, a silver também não pode ter mudado. Na primeira validação não há com o que comparar, e a linha fica com `ok` nulo.

In [0]:
# O QUE FAZ: compara as fotos de contagem desta validação com as da validação anterior e registra a
#   verificação 10 por entidade.
#
# FUNÇÕES NATIVAS:
# - spark.table(...).where(...).agg(F.max(...)).first()[0]: valor máximo como valor Python.
# - DataFrame.collect(): traz as linhas para o driver; aqui são poucas dezenas de linhas de
#   contagem (não dados de negócio), então é seguro.
# - dict comprehension {chave: valor for ...}: monta um dicionário em uma linha.
anteriores = spark.table(TABELA_RESULTADOS).where(F.col("verificacao").isin("foto_bronze", "foto_silver"))
ultima = anteriores.agg(F.max("verificado_em")).first()[0]

foto_antiga = {}
if ultima is not None:
    foto_antiga = {
        (r.entidade, r.verificacao): r.obtido
        for r in anteriores.where(F.col("verificado_em") == ultima).collect()
    }

for entidade in TODAS:
    bronze_agora = next(r[3] for r in resultados if r[0] == entidade and r[1] == "foto_bronze")
    silver_agora = next(r[3] for r in resultados if r[0] == entidade and r[1] == "foto_silver")
    bronze_antes = foto_antiga.get((entidade, "foto_bronze"))
    silver_antes = foto_antiga.get((entidade, "foto_silver"))
    if bronze_antes is None:
        registrar(entidade, "idempotencia_silver_estavel", None, silver_agora, "sem validação anterior")
    elif bronze_antes != bronze_agora:
        registrar(entidade, "idempotencia_silver_estavel", None, silver_agora, f"bronze mudou ({bronze_antes} -> {bronze_agora}); não comparável")
    else:
        registrar(entidade, "idempotencia_silver_estavel", silver_antes, silver_agora, "bronze igual à validação anterior")

## Gravação e resultado final

Grava todas as linhas em `quality.validacao_resultados` (uma tabela de histórico: cada validação acrescenta as suas) e falha o notebook se alguma verificação estiver reprovada, para que um job de orquestração (Fase 7) pare no problema.

In [0]:
# O QUE FAZ: grava os resultados, mostra o resumo e interrompe com erro se houver verificação reprovada.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(linhas, schema): cria um DataFrame a partir de uma lista de tuplas.
# - DataFrame.withColumns({...}): acrescenta colunas.
# - DataFrame.select(...) / .write.mode("append").saveAsTable(nome): acrescenta as linhas à tabela.
#     * append nunca apaga o histórico; o validacao_id separa uma validação da outra.
# - display(df): mostra o DataFrame como tabela interativa.
# - raise RuntimeError(texto): interrompe a execução com erro.
df = (
    spark.createDataFrame(
        resultados,
        "entidade string, verificacao string, esperado long, obtido long, ok boolean, detalhe string",
    )
    .withColumns({"validacao_id": F.lit(VALIDACAO_ID), "verificado_em": F.current_timestamp()})
    .select("validacao_id", "verificado_em", "entidade", "verificacao", "esperado", "obtido", "ok", "detalhe")
)
df.write.mode("append").saveAsTable(TABELA_RESULTADOS)

reprovadas = [r for r in resultados if r[4] is False]
total = [r for r in resultados if r[4] is not None]
print(f"\n{len(total) - len(reprovadas)} de {len(total)} verificações OK")
display(df.where(F.col("ok") == False).select("entidade", "verificacao", "esperado", "obtido", "detalhe"))  # noqa: E712

if reprovadas:
    raise RuntimeError(f"{len(reprovadas)} verificação(ões) reprovada(s): veja a tabela {TABELA_RESULTADOS}.")